In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# 1. Dataset paths
path = "/home/atharva-gawade/python-projects/cats_and_dogs_filtered (1)/cats_and_dogs_filtered"

train_dir = path + "/train"
val_dir = path + "/validation"

# 2. Load datasets
train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=(160, 160),
    batch_size=32,
    label_mode="binary"
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    val_dir,
    image_size=(160, 160),
    batch_size=32,
    label_mode="binary"
)

class_names = train_ds.class_names
print("Classes:", class_names)

# 3. Load pretrained MobileNetV2
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(160, 160, 3),
    include_top=False,
    weights="imagenet"
)

# Freeze pretrained layers
base_model.trainable = False

# 4. Create transfer learning model
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(160, 160, 3)),
    tf.keras.layers.Rescaling(1./127.5, offset=-1),
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

# 5. Compile model
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# 6. Train model
model.fit(
    train_ds,
    epochs=5,
    validation_data=val_ds
)

# 7. Evaluate model
loss, accuracy = model.evaluate(val_ds)
print("Validation Accuracy:", accuracy)

# 8. Display predictions
for images, labels in val_ds.take(1):
    predictions = model.predict(images[:5])

    for i in range(5):
        index = int(predictions[i][0] >= 0.5)

        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(
            "Actual: " + class_names[int(labels[i][0])] +
            "\nPredicted: " + class_names[index]
        )
        plt.axis("off")
        plt.show()